# Titanic Survival Prediction with Random Forest

In this notebook, we'll learn how to:
1. Load and explore the Titanic dataset
2. Clean and prepare the data
3. Train a Random Forest model
4. Save the model for use in a Flask application

**Goal:** Predict whether a passenger survived the Titanic disaster based on features like age, gender, and ticket class.

## Step 1: Import Required Libraries

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib  # For saving our model

print(" Libraries imported successfully!")

 Libraries imported successfully!


## Step 2: Load the Titanic Dataset

We'll load the famous Titanic dataset directly from a URL.

In [2]:
# Load the Titanic dataset from seaborn's built-in datasets
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Dataset loaded! Shape: {df.shape}")
print(f"\nWe have {df.shape[0]} passengers and {df.shape[1]} features.")

Dataset loaded! Shape: (891, 12)

We have 891 passengers and 12 features.


## Step 3: Explore the Data

Let's take a look at our data to understand what we're working with.

In [3]:
# Display first 5 rows
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Check the column names and data types
print(" Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\n Statistical Summary:")
df.describe()

 Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None


 Statistical Summary:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Total missing values: 866


## Step 4: Data Preprocessing (Cleaning the Data)

Before training our model, we need to:
1. Handle missing values
2. Convert categorical variables to numbers
3. Select the features we want to use

In [6]:
# Select features we'll use for prediction
# We'll keep it simple with these key features:
# - Pclass: Ticket class (1st, 2nd, 3rd)
# - Sex: Gender
# - Age: Age of passenger
# - SibSp: Number of siblings/spouses aboard
# - Parch: Number of parents/children aboard
# - Fare: Ticket price

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
target = 'Survived'

# Create a copy with only the columns we need
df_clean = df[features + [target]].copy()

print(f"Selected features: {features}")
print(f"Target variable: {target}")

Selected features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
Target variable: Survived


In [7]:
# Handle missing values

# Fill missing Age with the median age
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Fill missing Fare with the median fare
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())

print("Missing values handled!")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")

Missing values handled!
Remaining missing values: 0


In [8]:
# Convert 'Sex' to numbers (male=0, female=1)
# This is called "encoding" - machines understand numbers, not words!

df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})

print("Converted 'Sex' to numbers:")
print("   male -> 0")
print("   female -> 1")
print("\nCleaned data preview:")
df_clean.head()

Converted 'Sex' to numbers:
   male -> 0
   female -> 1

Cleaned data preview:


,Pclass,Sex,Age,SibSp,Parch,Fare,Survived
0,3,0,22.0,1,0,7.2500,0
1,1,1,38.0,1,0,71.2833,1
2,3,1,26.0,0,0,7.9250,1
3,1,1,35.0,1,0,53.1000,1
4,3,0,35.0,0,0,8.0500,0


## Step 5: Split Data into Training and Testing Sets

We'll use 80% of data for training and 20% for testing.

In [9]:
# Separate features (X) and target (y)
X = df_clean[features]
y = df_clean[target]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # 20% for testing
    random_state=42   # For reproducibility
)

print(f"Training set size: {len(X_train)} passengers")
print(f"Testing set size: {len(X_test)} passengers")

Training set size: 712 passengers
Testing set size: 179 passengers


## Step 6: Train the Random Forest Model 

Random Forest is a powerful algorithm that creates many decision trees and combines their predictions. It's like asking many experts and taking a vote!

In [10]:
# Create and train the Random Forest model
model = RandomForestClassifier(
    n_estimators=100,    # Number of trees in the forest
    max_depth=10,        # Maximum depth of each tree
    random_state=42      # For reproducibility
)

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Number of trees: {model.n_estimators}")

Model trained successfully!
Number of trees: 100


## Step 7: Evaluate the Model

In [11]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Did not survive', 'Survived']))

Model Accuracy: 80.45%

Classification Report:
                 precision    recall  f1-score   support

Did not survive       0.80      0.89      0.84       105
       Survived       0.81      0.69      0.74        74

       accuracy                           0.80       179
      macro avg       0.81      0.79      0.79       179
   weighted avg       0.80      0.80      0.80       179



In [12]:
# Let's see which features are most important
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (which features matter most):")
print(feature_importance)

Feature Importance (which features matter most):
  Feature  Importance
1     Sex    0.330312
5    Fare    0.260593
2     Age    0.212906
0  Pclass    0.095233
3   SibSp    0.055441
4   Parch    0.045516


## Step 8: Save the Model 

Now we'll save our trained model so we can use it in our Flask application!

In [13]:
# Save the model using joblib
model_filename = 'titanic_model.pkl'
joblib.dump(model, model_filename)

print(f"Model saved as '{model_filename}'")
print("\nYou can now use this model in your Flask application!")

Model saved as 'titanic_model.pkl'

You can now use this model in your Flask application!


## Step 9: Test Loading and Using the Saved Model

In [14]:
# Load the saved model (this is what our Flask app will do)
loaded_model = joblib.load(model_filename)

# Make a test prediction
# Example: 25-year-old female in 1st class, no siblings/spouse, no parents/children, fare of $100
test_passenger = [[1, 1, 25, 0, 0, 100]]  # [Pclass, Sex, Age, SibSp, Parch, Fare]

prediction = loaded_model.predict(test_passenger)
probability = loaded_model.predict_proba(test_passenger)

print("Test Prediction:")
print(f"Input: Pclass=1, Sex=Female, Age=25, SibSp=0, Parch=0, Fare=$100")
print(f"Prediction: {'Survived ' if prediction[0] == 1 else 'Did not survive '}")
print(f"Confidence: {max(probability[0]):.2%}")

Test Prediction:
Input: Pclass=1, Sex=Female, Age=25, SibSp=0, Parch=0, Fare=$100
Prediction: Survived 
Confidence: 97.00%


c:\Users\M1ne\Desktop\flask demo\flaskapp\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\M1ne\Desktop\flask demo\flaskapp\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
